# IPL 2026 Match Winner Prediction — Refactored ML Pipeline & Benchmark

This notebook documents the evolution from the initial `ipl.ipynb` prototype (which trained an uncalibrated 19-class `RandomForestClassifier` on raw One-Hot Encoded columns) to the production `v1.0.0` ML Pipeline:

1. **Canonical Entity Resolution**: Unifying rebranded franchises (`Royal Challengers Bangalore` $\rightarrow$ `Royal Challengers Bengaluru`, `Kings XI Punjab` $\rightarrow$ `Punjab Kings`, `Delhi Daredevils` $\rightarrow$ `Delhi Capitals`) and venues (`Bangalore` $\rightarrow$ `Bengaluru`, `Chandigarh` $\rightarrow$ `Mohali`).
2. **Leak-Free Rolling Feature Engineering**: Computing dynamic Elo ratings, Head-to-Head (H2H) win rates, rolling 5/10-match recent form, venue-specific chase bias, and toss-venue synergy strictly using matches prior to match $t$.
3. **Multi-Model Comparison**: Benchmarking **Logistic Regression**, **Random Forest**, **Gradient Boosting**, **XGBoost**, and a **Soft-Voting Hybrid Ensemble** with 5-Fold `GroupKFold` Cross-Validation.
4. **SHAP Explainability**: Extracting local and global feature attributions explaining *Why this prediction?*.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from ml.preprocessing import load_and_clean_dataset, build_training_dataset

df = load_and_clean_dataset(str(ROOT / 'IPL.csv'))
print(f'Cleaned matches: {len(df)} | Seasons: {df["season"].min()}-{df["season"].max()}')
df.head()

In [2]:
X_df, y_series, state_store = build_training_dataset(df, symmetric_augmentation=True)
print(f'Engineered training dataset shape: {X_df.shape}')
X_df.head()

In [3]:
from ml.train import train_and_export

metrics = train_and_export(ROOT / 'IPL.csv')
pd.DataFrame(metrics['models_compared'])[['model_name', 'accuracy', 'f1_score', 'roc_auc', 'brier_score', 'cv_roc_auc_mean']]